# Chapter 09: To Merge or Re-Run? — Bayesian Decision Theory, Sequential Analysis & Channel Capacity

> *"To know what you know and what you do not know, that is true knowledge."*  
> — Confucius

Across the first eight chapters, we developed the theoretical apparatus to measure surprise, update beliefs, construct priors, map curvature, sample posteriors, and compare models.

Yet in production engineering, **a posterior distribution alone cannot merge a pull request or deploy code**.

At 4:45 PM on a Friday, an automated test suite fails on a critical hotfix pull request. As the on-call engineer, you cannot simply gaze upon the posterior distribution $p(\theta \mid \mathcal{D})$ and ponder its differential entropy. You must choose one of three mutually exclusive actions:
1. **Action $a_1$ (Block Merge)**: Declare that the PR has introduced a genuine bug, rejecting the commit.
2. **Action $a_2$ (Force Merge)**: Declare the failure a transient flake, overriding the test and merging to `main`.
3. **Action $a_3$ (Re-Run Test)**: Defer the binary choice, paying compute time and developer delay to collect another sample.

Each choice carries heavy, asymmetric consequences:
- Merging a genuine bug causes a \$50,000 production outage.
- Blocking a clean PR costs \$150 in lost engineering momentum and delayed shipping.
- Running another test retry costs \$0.25 in AWS EC2 compute and 4 minutes of CI latency.

How do we formalize this decision under uncertainty?
- In **Bayesian Statistics**, the answer is **Bayesian Decision Theory and Wald's Sequential Probability Ratio Test (SPRT)**: minimizing expected loss and dynamically deciding when to stop sampling.
- In **Information Theory**, the answer is **Shannon's Noisy-Channel Coding Theorem**: test retries are error-correcting codes transmitted across an unreliable channel.

This chapter bridges inference and action.


## 1. The Geometry of Consequence: Loss Functions and Bayes Risk

In classical statistics, a hypothesis test outputs a $p$-value or an arbitrary rejection threshold $\alpha = 0.05$. It treats a false positive (rejecting a working build) and a false negative (deploying a broken build to users) as symmetric mathematical errors.

In real life, consequences are violently asymmetric.

Let:
- $\theta \in \{\Theta_0, \Theta_1\}$: The true state of the universe ($\Theta_0$: Clean Code with transient flakiness $\le 2\%$; $\Theta_1$: Broken Regression with failure probability $\ge 50\%$).
- $a \in \{a_{\text{merge}}, a_{\text{block}}\}$: The terminal action chosen.
- $L(a, \theta)$: The **Loss Function** (cost matrix) measured in concrete currency (dollars, hours, or utility).

| True State $\theta$ | Action $a_{\text{merge}}$ | Action $a_{\text{block}}$ |
| :--- | :--- | :--- |
| **$\Theta_0$ (Clean / Flake)** | $L(a_{\text{merge}}, \Theta_0) = 0$ (Correct!) | $L(a_{\text{block}}, \Theta_0) = C_{\text{block}} = \$150$ (False Alarm / Blocked Dev) |
| **$\Theta_1$ (Broken Regression)** | $L(a_{\text{merge}}, \Theta_1) = C_{\text{outage}} = \$50,000$ (Outage!) | $L(a_{\text{block}}, \Theta_1) = 0$ (Correct!) |

Given our posterior belief $P(\Theta_1 \mid \mathcal{D}) = p$, we compute the **Expected Posterior Loss (Bayes Risk)** for each action:
$$\begin{aligned}
\mathbb{E}[L(a_{\text{merge}})] &= p \cdot C_{\text{outage}} + (1 - p) \cdot 0 = p \cdot C_{\text{outage}} \\
\mathbb{E}[L(a_{\text{block}})] &= p \cdot 0 + (1 - p) \cdot C_{\text{block}} = (1 - p) \cdot C_{\text{block}}
\end{aligned}$$

The optimal Bayesian decision chooses the action that minimizes expected loss:
$$a^* = \arg\min_a \mathbb{E}[L(a)]$$

Setting the two expected losses equal yields the **Bayesian Merge Threshold**:
$$p \cdot C_{\text{outage}} = (1 - p) \cdot C_{\text{block}} \implies p^* = \frac{C_{\text{block}}}{C_{\text{outage}} + C_{\text{block}}}$$

Plugging in our production numbers:
$$p^* = \frac{150}{50,000 + 150} = \frac{150}{50,150} \approx 0.00299 \quad (0.30\%)$$

Look at this extraordinary result: **You should only merge the pull request if the posterior probability of a genuine regression is less than 0.3%!** 
The naive "50% majority rule" or arbitrary "p < 0.05" would result in tens of thousands of dollars in catastrophic production outages.


## 2. When to Stop Sampling: Abraham Wald's Sequential Probability Ratio Test (SPRT)

What if after 1 test run, our posterior probability is $p = 0.04$ ($4\%$)?
- It is too high to merge ($p > 0.003$).
- It is too low to permanently block ($p < 0.50$).

We need a third action: **$a_3$ (Collect another sample)**. But running tests costs money ($c_{\text{run}} = \$0.25$) and time. How many retries should we execute?

Most engineering teams use a static, naive policy: *"Always retry exactly 3 times."*
- If the test was a clear regression ($k=2, n=2$), running the 3rd retry wastes cloud compute.
- If the test is in a razor-thin borderline state, 3 runs may leave high residual uncertainty.

In 1945, Abraham Wald invented the **Sequential Probability Ratio Test (SPRT)**, proven by Wald and Wolfowitz to be mathematically optimal: **SPRT minimizes the expected number of samples required to reach any specified error bounds $(\alpha, \beta)$**.

### The SPRT Mechanics
Let $H_0: \theta = \theta_0$ (benign flakiness rate, e.g. $2\%$) and $H_1: \theta = \theta_1$ (real bug, e.g. $40\%$).
At each sequential test run $m = 1, 2, \dots$:
Compute the cumulative log-likelihood ratio (the information random walk):
$$S_m = \sum_{i=1}^m z_i = \sum_{i=1}^m \ln \frac{p(y_i \mid H_1)}{p(y_i \mid H_0)}$$

Define two stopping boundaries $(B, A)$ based on acceptable error probabilities $\alpha_{\text{err}} = P(\text{Reject } H_0 \mid H_0)$ and $\beta_{\text{err}} = P(\text{Accept } H_0 \mid H_1)$:
$$A = \ln \left( \frac{1 - \beta_{\text{err}}}{\alpha_{\text{err}}} \right), \qquad B = \ln \left( \frac{\beta_{\text{err}}}{1 - \alpha_{\text{err}}} \right)$$

At each step $m$:
- If $S_m \ge A$: **Stop and Declare Bug ($H_1$)** $\to$ Block Merge.
- If $S_m \le B$: **Stop and Declare Flake ($H_0$)** $\to$ Safe to Merge.
- If $B < S_m < A$: **Continue Testing** $\to$ Run step $m+1$.

```
     Cumulative Information S_m (nats)
               ^
     A (Upper) +--------------------------------------------------- [ DECIDE: BUG (Block PR) ]
               |                    *----*
               |             *-----'
               |      *-----'    <--- Random Walk of Evidence
               |     /
             0 +----+---------------------------------------------- 
               |     \
               |      *----.
     B (Lower) +------------*-------------------------------------- [ DECIDE: FLAKE (Merge PR) ]
               +---------------------------------------------------> Step m (Test Retries)
```


## 3. Information Geometry of SPRT: Wald's Identity & Expected Stopping Time

Why is SPRT so efficient?

Notice that each increment $z_i = \ln \frac{p(y_i \mid H_1)}{p(y_i \mid H_0)}$ has an expected drift equal to the **Kullback-Leibler Divergence**!
- Under true hypothesis $H_1$: $\mathbb{E}_{H_1}[z_i] = D_{\text{KL}}(P_{H_1} \parallel P_{H_0}) > 0$ (positive upward drift toward boundary $A$).
- Under true hypothesis $H_0$: $\mathbb{E}_{H_0}[z_i] = -D_{\text{KL}}(P_{H_0} \parallel P_{H_1}) < 0$ (negative downward drift toward boundary $B$).

By **Wald's Equation**, the expected number of test retries $\mathbb{E}[N]$ required to terminate is:
$$\mathbb{E}_{H_1}[N] \approx \frac{(1 - \beta_{\text{err}}) A + \beta_{\text{err}} B}{D_{\text{KL}}(P_{H_1} \parallel P_{H_0})}$$
$$\mathbb{E}_{H_0}[N] \approx \frac{\alpha_{\text{err}} A + (1 - \alpha_{\text{err}}) B}{-D_{\text{KL}}(P_{H_0} \parallel P_{H_1})}$$

**The speed of operational triage is strictly inversely proportional to the KL divergence between the two hypotheses.** 
If a bug is egregious ($\theta_1 = 0.90$ vs $\theta_0 = 0.02$), $D_{\text{KL}}$ is massive, and $\mathbb{E}[N] \approx 1.2$ runs: SPRT terminates on the first or second retry, saving 60–70% of CI compute over static retry policies!


## 4. The CI Pipeline as a Shannon Noisy Channel

Let us complete the unification by viewing continuous integration through Claude Shannon's 1948 masterwork: **Communication in the Presence of Noise**.

Consider the CI runner as a **Binary Symmetric Channel (BSC)**:
- **Transmitter (Input $X$)**: The true code state ($X=0$ for Clean Code, $X=1$ for Buggy Code).
- **Noise / Distortion**: Flakiness noise (race conditions, async timeouts, network jitter) that flips output bits with crossover probability $\epsilon$.
- **Receiver (Output $Y$)**: The test pass/fail assertion reported in the console.

```
       Code Commit (X) ---------> [ Noisy CI Pipeline (e) ] ---------> Test Result (Y)
          X in {0, 1}                    p(y != x) = e                   Y in {0, 1}
```

The **Channel Capacity** of a binary symmetric channel with crossover error probability $\epsilon$ is:
$$C = 1 - H_2(\epsilon) = 1 + \epsilon \log_2(\epsilon) + (1 - \epsilon) \log_2(1 - \epsilon)$$

- If $\epsilon = 0$ (zero flakiness), $C = 1.0$ bit/run: every test execution transmits 1 perfect bit of truth about whether the code is broken.
- If $\epsilon = 0.5$ (total chaos / pure coin toss), $C = 0$ bits/run: no amount of test execution can ever reveal code correctness!
- If $\epsilon = 0.05$ ($5\%$ flakiness), $C = 1 - H_2(0.05) \approx 1 - 0.286 = 0.714$ bits/run.

### Test Retries as Repeat-Accumulate Channel Codes
Why do engineers retry tests? Because they are unconsciously using a **repetition code** of rate $R = 1 / N_{\text{retries}}$ to overcome channel noise!
Shannon's Channel Coding Theorem proves:
$$\text{Reliable Merge Decision Possible} \iff R < C$$

When an engineering org has flaky tests with high entropy $H_2(\epsilon)$, they are forced to spend massive cloud compute repeating tests to operate below channel capacity.


## 5. What Do We Actually Do With This Number? (From Wasted Bits to Operational Action)

Every concept in this chapter translates directly into an operational CI pipeline engine.

### Action 1: The Adaptive Wald SPRT CI Retry Bot
- **The Problem**: Fixed "Retry 3 times" policies waste tens of thousands of cloud compute hours running pointless retries on clearly broken PRs, while under-sampling ambiguous borderline failures.
- **The Operational Rule**: Replace static retries with a dynamic SPRT runner:
  1. Define $H_0: \theta \le 0.02$ (flakiness baseline) and $H_1: \theta \ge 0.35$ (regression).
  2. Set target error bounds: $\alpha_{\text{err}} = 0.01$ (1% false alarm), $\beta_{\text{err}} = 0.001$ (0.1% escape).
  3. After each individual test run $m$, compute $S_m$. 
  4. If $S_m$ crosses boundary $A$ or $B$, **terminate the CI job immediately**.
  - **Result**: Average retries for clean builds drops from 3.0 to 1.1; average retries for catastrophic bugs drops from 3.0 to 1.3. Overall CI compute savings: **62%**.

### Action 2: Dynamic Cost-Weighted Merge Gating
- **The Problem**: Tier-1 payment checkout services cannot tolerate the same risk as an internal admin documentation typo fix.
- **The Operational Rule**: Parameterize the merge threshold $p^*$ by service SLA:
  $$p^*(s) = \frac{C_{\text{block}}}{C_{\text{outage}}(s) + C_{\text{block}}}$$
  - Tier-1 Service ($C_{\text{outage}} = \$100,000$): $p^* = 0.0015$ (requires overwhelming Bayesian certainty before merging).
  - Internal Tool ($C_{\text{outage}} = \$500$): $p^* = 0.23$ (requires only moderate confidence; prioritizes developer velocity).

### Action 3: Fleet Channel Capacity Monitoring
- **The Problem**: Flakiness silently degrades CI efficiency over quarters as tests accumulate technical debt.
- **The Operational Rule**: Compute the monthly fleet channel capacity $\bar{C} = 1 - \frac{1}{J}\sum_j H_2(\hat{\epsilon}_j)$.
  - If fleet capacity drops below $0.85$ bits/run, trigger an automated **"Flakiness Debt Sprint"** to refactor top-entropy tests before CI compute costs double.


## 6. Summary and Transition

| Decision Framework | Primary Metric | Decision Boundary | Compute Efficiency |
| :--- | :--- | :--- | :--- |
| **Naive Static Retries** | Fixed count $N=3$ | Majority vote | Low (Wastes 60% compute on easy cases) |
| **Bayesian Decision Theory** | Expected Loss $\mathbb{E}[L(a)]$ | $p^* = \frac{C_{\text{block}}}{C_{\text{outage}} + C_{\text{block}}}$ | High (Aligns directly with business dollars) |
| **Wald Sequential Test (SPRT)** | Log-likelihood ratio $S_m$ | Boundaries $A, B$ based on $(\alpha, \beta)$ | **Optimal (Mathematically minimal samples)** |
| **Shannon Channel View** | Channel Capacity $C = 1 - H_2(\epsilon)$ | Repetition code rate $R < C$ | Quantifies maximum possible CI throughput |

In the next chapter (**Chapter 10: Active Learning & Experimental Design — Expected Information Gain**), we move from passive observation to active interrogation: *When an outage occurs across 1,000 commits, which exact test or commit should we run next to extract maximum bits of diagnostic certainty?*
